# Models Pipeline

This notebook is the single restartable entry point. Reusable implementation lives in `src/`, while completed annual refits and diagnostics are cached by model ID and signature.

## Streamlined research stages

1. **Benchmarks:** `LASSO_20`, `LGBM_20`, `XGBOOST_20`, and `NN3_20` use the same ranked Core20 inputs.
2. **Characteristic breadth:** `LGBM_20`, `LGBM_40`, `LGBM_60`, `LGBM_80`, and `LGBM_100` test four predeclared mixed blocks of 20 characteristics.
3. **Firm dynamics:** `LGBM_20_LAG1`, `LGBM_20_LAG2`, `LGBM_40_LAG1`, and `LGBM_40_LAG2` test exact-calendar firm histories.
4. **Learned market representation:** `MLP_40` is the no-context control. `DEEPSET_40`, `DEEPSET_40_LAG1`, and `DEEPSET_40_DYNAMIC` add learned leave-one-out market context using the selected 40 characteristics.
5. **Secondary robustness:** the completed Core20 set models, induced-set Transformer, and validation-weighted hybrid remain secondary evidence.

## Fixed design decisions

- The target is decimal next-month excess return, `ret_exc_lead1m`.
- The universe is USA stocks with valid PERMNO and size group micro/small/large/mega; nano stocks are excluded.
- Characteristics are ranked within the complete eligible monthly cross-section and mapped to `[-1, 1]` before missing future returns are removed.
- Exact-calendar lags are joined by PERMNO. Missing prior months receive neutral values plus a zero availability flag.
- The rolling design uses 15 training years, 4 validation years, one untouched test year, and annual refits from 1999 through 2024.
- Validation MSE selects hyperparameters and early stopping. Test outcomes never affect fitting or model selection.
- Primary evaluation is pooled GKX OOS R-squared and equal-weighted D10-D1. Rank IC, calibration, robust R-squared, monotonicity, alternative portfolios, costs, and universe sensitivity are diagnostics.
- Constant-forecast months hold cash; PERMNO is only a deterministic tie-break when a genuine signal exists.
- Each model/refit has an independent signature and completion marker. Compatible results load without retraining or overwriting.

## 1. Runtime and project setup

The notebook file remains local and is edited in VS Code. When the selected kernel is Google Colab, this cell mounts Google Drive and uses the Drive copy of the project for source code, data, checkpoints, predictions, and diagnostics. With a normal local VS Code kernel, it uses the local Windows project folder instead. This storage choice does not change model definitions or experiment signatures.

In [1]:
import os
import sys
import importlib
from pathlib import Path

LOCAL_PROJECT_DIR = Path(r"C:\Users\sandh\OneDrive\Documents\Coding\FDS Project")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/FDS Project")

try:
    from google.colab import drive
except ImportError:
    RUNNING_IN_COLAB = False
    PROJECT_DIR = LOCAL_PROJECT_DIR
    RUNTIME = "Local VS Code kernel"
else:
    RUNNING_IN_COLAB = True
    # Safe under Run all: mounts Drive when needed and reuses an existing mount.
    drive.mount("/content/drive", force_remount=False)
    PROJECT_DIR = DRIVE_PROJECT_DIR
    RUNTIME = "Google Colab kernel in VS Code"

if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(
        f"Project folder was not found: {PROJECT_DIR}\n"
        "If this is Colab, confirm that Drive is mounted and that the folder name matches exactly."
    )
if not (PROJECT_DIR / "src").is_dir():
    raise FileNotFoundError(f"The project src folder was not found under: {PROJECT_DIR}")

os.chdir(PROJECT_DIR)
project_path = str(PROJECT_DIR)
# Always give this project priority over stale or similarly named packages.
sys.path = [path for path in sys.path if path != project_path]
sys.path.insert(0, project_path)
for module_name in tuple(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
importlib.invalidate_caches()

# Verify now, before any model configuration is evaluated.
import src
src_file = Path(src.__file__).resolve()
if PROJECT_DIR.resolve() not in src_file.parents:
    raise RuntimeError(f"Imported src from the wrong location: {src_file}")

print("Notebook file: local Models_Pipeline.ipynb")
print("Runtime:", RUNTIME)
print("Project files and outputs:", PROJECT_DIR)
print("Imported src from:", src_file)

Mounted at /content/drive
Notebook file: local Models_Pipeline.ipynb
Runtime: Google Colab kernel in VS Code
Project files and outputs: /content/drive/MyDrive/Colab Notebooks/FDS Project
Imported src from: /content/drive/MyDrive/Colab Notebooks/FDS Project/src/__init__.py


## 2. Experiment configuration

Usually this is the only cell to edit. Use a new `experiment_id` after changing the universe, feature definition, target, preprocessing, or rolling schedule. The same experiment can safely be rerun or extended with new registered models.

In [2]:
if 'PROJECT_DIR' not in globals():
    raise RuntimeError('Run the Runtime and project setup cell first, or use Run all.')

from src.config import ExperimentConfig
from src.models import MODEL_REGISTRY

DATA_PATH = PROJECT_DIR / 'jkp_USA_100chars_1980_2024.parquet'
OUTPUT_DIR = PROJECT_DIR / 'model_runs'
if not DATA_PATH.is_file():
    raise FileNotFoundError(f'Raw data file not found: {DATA_PATH}')

STAGE_MODELS = {
    'stage_1_benchmarks': ('LASSO_20', 'LGBM_20', 'XGBOOST_20', 'NN3_20'),
    'stage_2_expansion': ('LGBM_20', 'LGBM_40', 'LGBM_60', 'LGBM_80', 'LGBM_100'),
    'stage_3_firm_dynamics': (
        'LGBM_20_LAG1', 'LGBM_20_LAG2',
        'LGBM_40_LAG1', 'LGBM_40_LAG2',
    ),
    'stage_4_market_representation': (
        'MLP_40', 'DEEPSET_40',
        'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC',
    ),
    'stage_5_secondary_robustness': (
        'DEEPSET_20', 'DEEPSET_20_LAG1', 'DEEPSET_20_DYNAMIC',
    ),
    'stage_6_hybrid': (
        'HYBRID_LGBM20_DEEPSET20', 'HYBRID_MLP40_DEEPSET40',
    ),
}
STAGE_MODELS['remaining_models'] = (
    'DEEPSET_40_LAG1',
    'DEEPSET_40_DYNAMIC',
    'HYBRID_MLP40_DEEPSET40',
)
STAGE_MODELS['all_completed'] = tuple(dict.fromkeys(
    model for stage_name, stage in STAGE_MODELS.items()
    if stage_name != 'remaining_models'
    for model in stage
))

RUN_STAGE = 'all_completed'  # Final pass: reuse saved fits and rebuild current diagnostics for every retained model.

CONFIG = ExperimentConfig(
    experiment_id='core20_benchmarks_v1',
    project_dir=PROJECT_DIR,
    data_path=DATA_PATH,
    output_dir=OUTPUT_DIR,
    selected_models=STAGE_MODELS[RUN_STAGE],
    seed=42,
    use_gpu=True,
)
CONFIG.validate()

import torch
if CONFIG.use_gpu and not torch.cuda.is_available():
    raise RuntimeError('GPU requested but unavailable. In Colab choose Runtime > Change runtime type > T4 GPU, then reconnect.')
print('Torch device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('Stage:', RUN_STAGE)
print('Selected models:', list(CONFIG.selected_models))
print('Run directory:', CONFIG.run_dir)

Torch device: NVIDIA A100-SXM4-40GB
Stage: new_models_final
Selected models: ['MLP_40', 'LGBM_20_LAG2', 'LGBM_40_LAG2', 'DEEPSET_40', 'DEEPSET_40_LAG1', 'DEEPSET_40_DYNAMIC']
Run directory: /content/drive/MyDrive/Colab Notebooks/FDS Project/model_runs/core20_benchmarks_v1


## 3. Preflight checks

This verifies the Drive project files, model registry and deterministic feature-construction checks without loading the full panel. The runner loads and prepares the data once in the next section.

In [3]:
import unittest
from src.self_checks import run_framework_self_checks

required_files = (
    DATA_PATH,
    PROJECT_DIR / 'src' / 'config.py',
    PROJECT_DIR / 'src' / 'models.py',
    PROJECT_DIR / 'src' / 'runner.py',
    PROJECT_DIR / 'src' / 'self_checks.py',
    PROJECT_DIR / 'tests' / 'test_pipeline.py',
)
missing_files = [str(path) for path in required_files if not path.is_file()]
if missing_files:
    raise FileNotFoundError(f'Required Drive project files are missing: {missing_files}')
unknown_models = sorted(set(CONFIG.selected_models) - set(MODEL_REGISTRY))
if unknown_models:
    raise ValueError(f'Selected models are not registered: {unknown_models}')

run_framework_self_checks()
suite = unittest.defaultTestLoader.discover(str(PROJECT_DIR / 'tests'))
test_result = unittest.TextTestRunner(verbosity=1).run(suite)
if not test_result.wasSuccessful():
    raise RuntimeError('Pipeline unit tests failed.')
print('Drive project files: PASS')
print('Selected model registry: PASS')
print('Framework self-checks: PASS')
print('Pipeline unit tests: PASS')

Drive project files: PASS
Selected model registry: PASS
Framework self-checks: PASS


## 4. Run or resume

This is safe to rerun. Compatible completed refits load, incomplete work resumes or reruns, pooled files rebuild from refit outputs, and metrics and portfolios refresh.

In [ ]:
# Run one model at a time so pandas never constructs the union of every
# model's lagged feature blocks in memory. Artifacts and resume logic are unchanged.
from dataclasses import replace
import gc
import torch
from src.runner import ExperimentRunner

for model_id in CONFIG.selected_models:
    print(f'\n=== {model_id} ===')
    model_config = replace(CONFIG, selected_models=(model_id,))
    ExperimentRunner(model_config).run()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# Restore the full selected-model view for comparison and later cells.
runner = ExperimentRunner(CONFIG)
comparison = runner._cumulative_comparison()
display(comparison)


=== MLP_40 ===
Device: cuda

MLP_40 [8dcf57348e2a3d38]
    epoch 001: train_mse=0.02259312 validation_mse=0.02747422
    epoch 002: train_mse=0.02240442 validation_mse=0.02737490
    epoch 003: train_mse=0.02233845 validation_mse=0.02731829
    epoch 004: train_mse=0.02232889 validation_mse=0.02738454
    epoch 005: train_mse=0.02230294 validation_mse=0.02731594
    epoch 006: train_mse=0.02225829 validation_mse=0.02723521
    epoch 007: train_mse=0.02228567 validation_mse=0.02722096
    epoch 008: train_mse=0.02227572 validation_mse=0.02723581
    epoch 009: train_mse=0.02226283 validation_mse=0.02729002
    epoch 010: train_mse=0.02223267 validation_mse=0.02723448
    epoch 011: train_mse=0.02222668 validation_mse=0.02730353
    epoch 012: train_mse=0.02224147 validation_mse=0.02724423
    epoch 013: train_mse=0.02221614 validation_mse=0.02722066
    epoch 014: train_mse=0.02220407 validation_mse=0.02718074
    epoch 015: train_mse=0.02221152 validation_mse=0.02740931
    epoch 016:

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  2014: saved 44,142 predictions


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  2015: saved 45,460 predictions


...

  2022: saved 55,599 predictions


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  2023: saved 52,611 predictions


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


  2024: saved 48,561 predictions

=== DEEPSET_40 ===
Device: cuda

DEEPSET_40 [ae11c35194b562c2]
    epoch 001: train_mse=0.02258754 validation_mse=0.02747413
    epoch 002: train_mse=0.02240114 validation_mse=0.02737524
    epoch 003: train_mse=0.02233941 validation_mse=0.02731537
    epoch 004: train_mse=0.02233246 validation_mse=0.02738623
    epoch 005: train_mse=0.02230478 validation_mse=0.02731395
    epoch 006: train_mse=0.02225889 validation_mse=0.02723196
    epoch 007: train_mse=0.02228281 validation_mse=0.02722001
    epoch 008: train_mse=0.02227467 validation_mse=0.02724009
    epoch 009: train_mse=0.02226247 validation_mse=0.02729257
    epoch 010: train_mse=0.02223193 validation_mse=0.02723706
    epoch 011: train_mse=0.02221997 validation_mse=0.02730520
    epoch 012: train_mse=0.02224099 validation_mse=0.02724516
    epoch 013: train_mse=0.02221440 validation_mse=0.02722043
    epoch 014: train_mse=0.02219798 validation_mse=0.02717788
    epoch 015: train_mse=0.02220315

## 5. Reload saved comparison without training

In [ ]:
import pandas as pd

comparison_path = CONFIG.run_dir / 'model_comparison.csv'
if comparison_path.exists():
    display(pd.read_csv(comparison_path))
else:
    print('No completed model comparison exists yet.')

## 7. Portfolio implementability robustness

This separate cached stage reads each pooled prediction file once and never loads model weights. It evaluates the 10% tail portfolio under full/ex-microcap universes, equal/value weighting, and fixed proportional transaction-cost scenarios.

In [ ]:
from src.portfolio_robustness import run_portfolio_robustness

portfolio_robustness = run_portfolio_robustness(
    CONFIG.run_dir,
    model_ids=tuple(
        model for model in CONFIG.selected_models
        if not model.endswith('_CALIBRATED')
    ),
)
display(portfolio_robustness)

## 8. Paired model tests

Run this only after the required models have final monthly diagnostic and portfolio-variant files. The default is strict: a missing planned pair raises an error rather than silently producing an incomplete comparison.

In [ ]:
from src.model_comparison import run_stage2_comparisons

stage2_comparison = run_stage2_comparisons(CONFIG.run_dir, seed=CONFIG.seed)
display(stage2_comparison)

## 9. Final completion checks and frozen outputs

In [ ]:
# Recreate the lightweight runner in case the kernel was restarted or this
# cell is run independently. This does not call runner.run() or train models.
from src.runner import ExperimentRunner
runner = ExperimentRunner(CONFIG)
final_comparison = runner._cumulative_comparison()
expected_models = set(CONFIG.selected_models)
completed_models = set(final_comparison.loc[
    final_comparison['diagnostics_version'].eq(runner.DIAGNOSTICS_VERSION), 'model_id'
])
missing_current = sorted(expected_models - completed_models)
if missing_current:
    raise RuntimeError(f'Models missing current diagnostics: {missing_current}')
expected_oos_months = 12 * (
    CONFIG.universe.end_year - CONFIG.universe.start_year
    - CONFIG.windows.train_years - CONFIG.windows.validation_years + 1
)
selected_comparison = final_comparison.set_index('model_id').loc[list(expected_models)]
if selected_comparison['n_months'].ne(expected_oos_months).any():
    raise RuntimeError('At least one model does not cover the complete OOS portfolio calendar.')
if (selected_comparison['n_signal_months'] + selected_comparison['n_no_signal_months']).ne(expected_oos_months).any():
    raise RuntimeError('At least one model has an incomplete signal-availability accounting.')
required_columns = [
    'robust_oos_r2', 'mean_monthly_rank_ic', 'n_no_signal_months',
    'mean_monthly_calibration_slope', 'tail_5pct_sharpe',
    'tail_10pct_sharpe', 'tail_20pct_sharpe', 'rank_weighted_sharpe',
    'tail_10pct_missing_return_stress_annualized_return',
]
missing_values = final_comparison.set_index('model_id').loc[list(expected_models), required_columns].isna()
if missing_values.any().any():
    print('Legitimate undefined diagnostics require review:')
    display(missing_values[missing_values.any(axis=1)])
else:
    print('All selected models have current complete diagnostics.')
expected_robustness_rows = 4 * sum(
    not model.endswith('_CALIBRATED') for model in expected_models
)
if len(portfolio_robustness) != expected_robustness_rows:
    raise RuntimeError(
        f'Expected {expected_robustness_rows} robustness rows, got {len(portfolio_robustness)}.'
    )
from src.model_comparison import DEFAULT_MODEL_PAIRS
if len(stage2_comparison) != len(DEFAULT_MODEL_PAIRS):
    raise RuntimeError('Paired model comparison is incomplete.')
print('Portfolio robustness and paired model comparisons are complete.')

## Adding a model later

Add a `ModelSpec` and trainer in `src/models.py`, then add its ID to `selected_models`. Existing completed model signatures remain unchanged. Add a focused self-check for every new architecture before starting a full rolling run.